In [1]:
import logging

import numpy as np
import polars as pl

import torch

from datasets import load_dataset

from sklearn.feature_extraction.text import TfidfVectorizer

from transformers import AutoModel, AutoModelForSeq2SeqLM, AutoTokenizer, pipeline
from sentence_transformers import SentenceTransformer, util

from transformers import logging as hf_logging
from transformers.utils.logging import disable_progress_bar

DATA = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
data = pl.read_csv(DATA)

In [2]:
hf_logging.set_verbosity_error()

logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)

disable_progress_bar()

# Introduction to Hugging Face transformers and datasets

In [3]:
# Load train.csv using the Hugging Face datasets library (do not use pandas). 
# Use the .map() function to create a new column called combined_text that 
# concatenates the prompt and A columns with a space in between. 
# E.g., prompt_text A_text. What is the exact character length 
# (total number of string characters using Python's len() function, 
# NOT the number of tokens) of the combined_text string for the row at index 51? 
# Note: We follow zero-indexing here.

dataset = load_dataset("csv", data_files=DATA, split="train")
dataset = dataset.map(lambda row: {"combined_text": f"{row["prompt"]} {row["A"]}"})

char_length = len(dataset[51]["combined_text"])

print(f"Character length at index 51: {char_length}")

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Character length at index 51: 614


In [4]:
# Initialize the bert-base-uncased tokenizer. Look at the tokenizer's configuration 
# properties: what is the exact total vocabulary size (the maximum number of 
# unique subword tokens the model knows) hardcoded into this tokenizer?  

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

vocab_size = tokenizer.vocab_size

print(f"Vocabulary size of 'bert-base-uncased' tokenizer: {vocab_size}")

Vocabulary size of 'bert-base-uncased' tokenizer: 30522


In [5]:
# Transformers rely on special tokens to understand sentence boundaries. 
# Using the bert-base-uncased tokenizer from the previous step, 
# extract the exact integer ID assigned to the [SEP] (Separator) token.  

sep_id = tokenizer.sep_token_id

print(f"[SEP] token ID of 'bert-base-uncased' tokenizer: {sep_id}")

[SEP] token ID of 'bert-base-uncased' tokenizer: 102


In [6]:
# Using the bert-base-uncased tokenizer, tokenize the entire prompt column of the 
# train dataset simultaneously. Set padding='max_length', truncation=True, 
# max_length=128, and return_tensors='pt' (PyTorch tensors). 

# What is the exact geometric shape (dimensions) of the resulting input_ids tensor?

prompts_list = [str(text) for text in dataset["prompt"]]

tokenized_outputs = tokenizer(
    prompts_list,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

tensor_shape = tokenized_outputs["input_ids"].shape

print(f"input_ids shape: {list(tensor_shape)}")

input_ids shape: [2000, 128]


# BERT/RoBERTa Architecture & Attention Mechanisms 

In [7]:
# A standard bert-base-uncased model has a hidden embedding size of 
# 768 dimensions and uses exactly 12 attention heads in each layer. 

# In Transformer architecture, the hidden size is divided equally among 
# the attention heads. What is the exact dimensionality (size) of each individual attention head?  

hidden_size = 768
num_heads = 12

print(f"Dimenstionality of each Attention Head: {hidden_size // num_heads}")

Dimenstionality of each Attention Head: 64


In [8]:
# Load the bert-base-uncased model using AutoModel.from_pretrained(). 
# Tokenize the prompt from row ID 0 using the tokenizer's default settings 
# (do not apply any manual padding or truncation). Pass this tokenized input 
# through the model. Look at the output object. 

# What is the exact shape of the last_hidden_state tensor returned? 

# Note: We follow zero-indexing here.

model = AutoModel.from_pretrained("bert-base-uncased")

prompt_row_0 = str(dataset[0]["prompt"])
inputs = tokenizer(prompt_row_0, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

hidden_state_shape = outputs.last_hidden_state.shape

print(f"last_hidden_state shape: {list(hidden_state_shape)}")

last_hidden_state shape: [1, 31, 768]


In [9]:
# Using the last_hidden_state tensor from the previous question, 
# extract the embedding vector representing the [CLS] token 
# (which is always the token at index 0). What is the sum of 
# the first 5 float values in this [CLS] vector? (Round your answer to 4 decimal places).  

cls_embedding = outputs.last_hidden_state[0, 0, :]
cls_sum = cls_embedding[:5].sum().item()

print(f"Sum of first 5 [CLS] values: {cls_sum:.4f}")

Sum of first 5 [CLS] values: -1.2001


In [10]:
# Load bert-base-uncased with the parameter output_attentions=True. 
# Tokenize the exact string "Light-ion fusion is a technique." 
# (ensuring you set return_tensors='pt') and pass it through the model. 
# Extract the attention matrix for the last layer (index -1) 
# and the first attention head (head index 0). 

# What is the exact attention weight (a float value) that the [CLS] token 
# (token index 0) pays to the word fusion (you will need to find the specific token index 
# for fusion in the input_ids)? (Round your answer to 4 decimal places).  

model_with_attention = AutoModel.from_pretrained("bert-base-uncased", output_attentions=True)

text = "Light-ion fusion is a technique."
inputs_attn = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs_attn = model_with_attention(**inputs_attn)

attentions = outputs_attn.attentions

tokens = tokenizer.convert_ids_to_tokens(inputs_attn["input_ids"][0])
fusion_index = tokens.index("fusion")

# Extract the specific attention weight:
# - Last layer: index -1
# - Batch size: index 0
# - First attention head: index 0
# - Query token ([CLS]): index 0
# - Key token ("fusion"): fusion_index
attention_weight = attentions[-1][0, 0, 0, fusion_index].item()

print(f"Attention weight the [CLS] token pays to the word 'fusion'): {attention_weight:.4f}")

Attention weight the [CLS] token pays to the word 'fusion'): 0.1025


# Context-Aware Embeddings 

In [11]:
# Initialize the sentence-transformers/all-MiniLM-L6-v2 model. Use the model's 
# .encode() method to generate embeddings for both the prompt and Option B for row ID 0. 
# Calculate the cosine similarity between these two vectors specifically using the 
# sentence_transformers.util.cos_sim() function. What is the resulting similarity score 
# rounded to 4 decimal places? Note: We follow zero-indexing here.

st_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

prompt_text = str(dataset[0]["prompt"])
option_b_text = str(dataset[0]["B"])

embedding_prompt = st_model.encode(prompt_text)
embedding_b = st_model.encode(option_b_text)

cos_sim = util.cos_sim(embedding_prompt, embedding_b).item()

print(f"Cosine Similarity between prompt and option B for : {cos_sim:.4f}")

Cosine Similarity between prompt and option B for : 0.7658


In [12]:
# Build two complete ranking pipelines evaluating every row in train.csv.

# Pipeline 1: Use the TF-IDF cosine similarity approach from Milestone 1.

# Pipeline 2: Use the sentence-transformers/all-MiniLM-L6-v2 model to 
# generate embeddings for the prompt and all five options. 
# Rank options using cosine similarity to form Top-3 predictions.

# First, what is the final MAP@3 score of the all-MiniLM-L6-v2 
# pipeline across the entire training set? 

# Second, count the number of questions for which the correct answer 
# is NOT present in the TF-IDF Top-3 predictions BUT IS present 
# in the MiniLM Top-3 predictions. What is this exact resulting count?  

def map3(actual, predicted):
    if len(predicted) > 3:
        predicted = predicted[:3]
        
    score = 0.0
    num_hits = 0.0
    
    for i, p in enumerate(predicted):
        if p == actual and p not in predicted[:i]:
            num_hits += 1.0
            score += num_hits / (i + 1.0)
            
    return score

# Training TF-IDF vectorizer and computing cosine similarity scores with numpy  from milestone-1
text_columns = ["prompt", "A", "B", "C", "D", "E"]

all_text = []
for col in text_columns:
    all_text.extend(data[col].drop_nulls().to_list())

vectorizer = TfidfVectorizer(stop_words="english")
vectorizer.fit(all_text)

prompt_tfidf = vectorizer.transform(data["prompt"].to_list())
A_tfidf = vectorizer.transform(data["A"].to_list())
B_tfidf = vectorizer.transform(data["B"].to_list())
C_tfidf = vectorizer.transform(data["C"].to_list())
D_tfidf = vectorizer.transform(data["D"].to_list())
E_tfidf = vectorizer.transform(data["E"].to_list())

def rowwise_cosine_sim(mat1, mat2):
    return np.array(mat1.multiply(mat2).sum(axis=1)).flatten()

sim_A = rowwise_cosine_sim(prompt_tfidf, A_tfidf)
sim_B = rowwise_cosine_sim(prompt_tfidf, B_tfidf)
sim_C = rowwise_cosine_sim(prompt_tfidf, C_tfidf)
sim_D = rowwise_cosine_sim(prompt_tfidf, D_tfidf)
sim_E = rowwise_cosine_sim(prompt_tfidf, E_tfidf)

all_sims_tfidf = np.vstack([sim_A, sim_B, sim_C, sim_D, sim_E]).T

top_3_idx_tfidf = np.argsort(all_sims_tfidf, axis=1)[:, ::-1][:, :3]

options_array = np.array(["A", "B", "C", "D", "E"])
top_3_tfidf_preds = options_array[top_3_idx_tfidf]

st_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

prompts = data["prompt"].to_list()
options_lists = [
    data["A"].to_list(),
    data["B"].to_list(),
    data["C"].to_list(),
    data["D"].to_list(),
    data["E"].to_list()
]

prompt_embs = st_model.encode(prompts, convert_to_tensor=True, batch_size=32)

all_sims_minilm = []

for i in range(5):
    opt_embs = st_model.encode(options_lists[i], convert_to_tensor=True, batch_size=32)
    sims = torch.nn.functional.cosine_similarity(prompt_embs, opt_embs).cpu().numpy()
    all_sims_minilm.append(sims)

all_sims_minilm = np.vstack(all_sims_minilm).T

top_3_idx_minilm = np.argsort(all_sims_minilm, axis=1)[:, ::-1][:, :3]
top_3_minilm_preds = options_array[top_3_idx_minilm]

true_answers = data["answer"].to_list()

minilm_scores = []
improvement_count = 0

for i in range(len(true_answers)):
    correct_answer = true_answers[i]
    
    tfidf_top3 = top_3_tfidf_preds[i].tolist()
    minilm_top3 = top_3_minilm_preds[i].tolist()
    
    minilm_scores.append(map3(correct_answer, minilm_top3))
    
    if (correct_answer not in tfidf_top3) and (correct_answer in minilm_top3):
        improvement_count += 1

final_map3 = np.mean(minilm_scores)

print(f"Final MAP@3 score of all-MiniLM-L6-v2 pipeline: {final_map3:.4f}")
print(f"Number of questions missed by TF-IDF Top-3 BUT found in MiniLM Top-3: {improvement_count}")

Final MAP@3 score of all-MiniLM-L6-v2 pipeline: 0.4231
Number of questions missed by TF-IDF Top-3 BUT found in MiniLM Top-3: 575


# Zero-shot classification concepts 

In [13]:
# Initialize the Hugging Face pipeline for "zero-shot-classification" 
# (it will default to facebook/bart-large-mnli). For the prompt of the 
# 2nd row (index 1), pass Options A, B, and C as the candidate_labels. 
# What is the probability score given to the top-ranked option? (Round to 4 decimal places).

zero_shot_classifier = pipeline("zero-shot-classification")

prompt_idx1 = str(data["prompt"].to_list()[1])
candidate_labels = [
    str(data["A"].to_list()[1]),
    str(data["B"].to_list()[1]),
    str(data["C"].to_list()[1])
]

result_single = zero_shot_classifier(prompt_idx1, candidate_labels)

top_score = result_single['scores'][0]

print(f"Top-ranked probability score: {top_score:.4f}")

Top-ranked probability score: 0.4575


In [14]:
# Run the exact same zero-shot classification as the previous question, 
# but this time pass the argument multi_label=True. 

# What is the absolute difference between the sum of the 3 probabilities 
# in the previous question (which uses Softmax) and 
# the sum of the 3 probabilities in this question (which uses independent Sigmoids)?

result_multi = zero_shot_classifier(prompt_idx1, candidate_labels, multi_label=True)

sum_single = sum(result_single['scores']) # Uses Softmax (should equal ~1.0)
sum_multi = sum(result_multi['scores']) # Uses independent Sigmoids

print(f"Absolute difference between sums: {abs(sum_single - sum_multi):.4f}")

Absolute difference between sums: 0.9995


In [15]:
# Let's try Generative AI instead of Classification. 

# Load a Small Language Model like google/flan-t5-small 
# using the Hugging Face pipeline("text2text-generation"). 
# Construct the following exact string for row index 0: 
# "Question: [prompt]. Is the correct answer A: [A] or B: [B]? Answer with just the letter A or B." 

# Pass this string to the pipeline, setting max_new_tokens=5. 
# What is the exact string output returned by the model? 

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

prompt_idx0 = str(data["prompt"].to_list()[0])
opt_A_idx0 = str(data["A"].to_list()[0])
opt_B_idx0 = str(data["B"].to_list()[0])

input_string = f"Question: {prompt_idx0}. Is the correct answer A: {opt_A_idx0} or B: {opt_B_idx0}? Answer with just the letter A or B."

inputs = tokenizer(input_string, return_tensors="pt")

outputs = model.generate(**inputs, max_new_tokens=5)

outout_string = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(f"Output string returned by the SLM: {outout_string}")

Output string returned by the SLM: B
